In [1]:
import warnings
from pathlib import Path

import prism

from imagematerials.concepts import knowledge_graph
from imagematerials.factory import ModelFactory
from imagematerials.maintenance import Maintenance
from imagematerials.model import GenericMainModel, GenericMaterials, GenericStocks
from imagematerials.util import (
    export_to_netcdf, import_from_netcdf, rebroadcast_prep_data,
    read_climate_policy_config, read_circular_economy_config
)
from imagematerials.vehicles import (
    preprocess
)


In [2]:
base_dir = "../data/raw"
climate_policy_scenario_dir = Path(base_dir) / 'SSP2'
circular_economy_scenario_dirs = {"slow": Path(base_dir) / 'circular_economy_scenarios' / 'slow'}
prep_fp = Path("prep_vema.nc")
# note: configuration currently only works when no prep file is saved

In [ ]:
import pandas as pd
from imagematerials.vehicles.modelling_functions import interpolate

lifetimes_vehicles: pd.DataFrame = pd.read_csv("../data/raw/vehicles/SSP2_CP\lifetimes_years.csv", index_col=[
        0, 1])
lifetimes_vehicles = lifetimes_vehicles.rename_axis('mode', axis=1).stack()
lifetimes_vehicles = lifetimes_vehicles[(lifetimes_vehicles.T != 0)]
lifetimes_vehicles = lifetimes_vehicles.unstack(['mode', 'data'])
lifetimes_vehicles = interpolate(pd.DataFrame(lifetimes_vehicles))
lifetimes_vehicles

In [ ]:
increase_config = {
    'car': 22.5,
    'LCV': 10,
    'MFT': 10,
    'HFT': 10,
    'rail_reg': 38,
    'rail_hst': 38,
    'rail_freight': 38,
    'sea_shipping_small': 33,
    'sea_shipping_med': 33,
    'sea_shipping_large': 33,
    'sea_shipping_vl': 33,
    'air_pas': 10,
    'air_freight': 10,
    'bicycle': 20,
    'inland_shipping': 33,
    'midi_bus': 20,
    'reg_bus': 20,
}
base_year = 2019
target_year = 2060

lifetimes_vehicles_inc = lifetimes_vehicles[lifetimes_vehicles.index <= base_year].copy()
lifetimes_vehicles_inc.loc[target_year] = lifetimes_vehicles_inc.loc[base_year]

for mode, increase in increase_config.items():
    col = (mode, 'mean')
    if col in lifetimes_vehicles_inc.columns:
        base_val = lifetimes_vehicles_inc.loc[base_year, col]
        lifetimes_vehicles_inc.loc[target_year, col] = base_val * (1 + increase / 100)
    else:
        print(f"Missing mode: {col}")
lifetimes_vehicles_inc = interpolate(pd.DataFrame(lifetimes_vehicles_inc))

In [ ]:

df = lifetimes_vehicles




# Apply the lifetime increases for the target year
for mode, percent_increase in increase_config.items():
    col = (mode, 'mean')
    if col in df.columns:
        base_val = df.loc[base_year, col]
        print(base_val)
        increase_factor = 1 + (percent_increase / 100)
        print(increase_factor)
        df.loc[target_year, col] = base_val * increase_factor
    else:
        print(f"Column {col} not found in DataFrame")

# Re-interpolate over all years for the 'mean' data only
for mode in df.columns.levels[0]:
    if ('mean' in df[mode].columns):
        df[mode, 'mean'] = df[mode, 'mean'].interpolate(method='linear')

df#[target_year,('car','mean')]

In [3]:
if not prep_fp.is_file():
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        climate_policy_config = read_climate_policy_config(climate_policy_scenario_dir)
        circular_economy_config = read_circular_economy_config(circular_economy_scenario_dirs)
        orig_prep_data = preprocess(base_dir, climate_policy_config, circular_economy_config)
    export_to_netcdf(orig_prep_data, prep_fp)
prep_data = import_from_netcdf(prep_fp)
prep_data["weights"] = prep_data.pop("vehicle_weights")

KeyError: 'config_file_path'

In [ ]:
# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

In [ ]:
# Define the coordinates of all dimensions.
Region = list(prep_data["stocks"].coords["Region"].values)
Time = [t for t in complete_timeline]
Cohort = Time
Type = list(prep_data["stocks"].coords["Type"].values)
material = list(prep_data["material_fractions"].coords["material"].values)

# Create
# main_model_normal = GenericMainModel(
#     complete_timeline, Region=Region, Time=Time, Cohort=Cohort, Type=Type, prep_data=prep_data,
#     compute_materials=True, compute_battery_materials=False, compute_maintenance_materials=False, 
#     material=material, knowledge_graph=knowledge_graph)

In [ ]:
new_prep_data = rebroadcast_prep_data(prep_data, knowledge_graph, dim="Type", output_coords=prep_data["shares"].coords["Type"].values)
new_prep_data = rebroadcast_prep_data(new_prep_data, knowledge_graph, dim="Region", output_coords=prep_data["shares"].coords["Region"].values)
new_prep_data["knowledge_graph"] = knowledge_graph

In [ ]:
# main_model_normal.simulate(simulation_timeline)

In [ ]:
main_model_factory = ModelFactory(
    new_prep_data, complete_timeline
    ).add(GenericStocks
    ).add(GenericMaterials
    ).finish()

In [ ]:
main_model_factory.simulate(simulation_timeline)